# Phase 2 — LoRA-QAT (INT8 & INT4) — Google Colab

**Goal:** measure the cheapest QAT variant. Quantize the base model first, then attach small LoRA adapters (rank 16, ~1–2% of total parameters) to recover quality.

**Inputs:** `/content/drive/MyDrive/sqat-baseline/results/baseline/fp32_logits.pt`

**Outputs (saved at the end to `/content/drive/MyDrive/sqat-lora-qat/`):**
- `models/checkpoints/lora_qat_int{4,8}_adapter/` — PEFT adapter dirs
- `results/lora_qat_int{4,8}/training_results.json`
- `results/lora_qat_inference.json`
- `results/logs/lora_qat_int{4,8}/per_step_loss.jsonl` — micro view
- `results/logs/lora_qat_int{4,8}/training_steps.jsonl` — macro view
- `data_samples/{train,val,test}_sample.pt`
- `results/metric_summary.csv`

**Estimated time on Colab T4:** ~2.5 h total (fewer trainable params)."

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

WORKING_DIR  = "/content"
REPO_DIR     = f"{WORKING_DIR}/scheduled-qat-for-slm"
GITHUB_URL   = "https://github.com/JpCurada/scheduled-qat-for-slm.git"
BASELINE_DIR = "/content/drive/MyDrive/sqat-baseline/results/baseline"
DRIVE_ROOT   = "/content/drive/MyDrive/sqat-lora-qat"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

FP32_LOGITS = Path(BASELINE_DIR) / "fp32_logits.pt"
assert FP32_LOGITS.exists(), f"FP32 logits not found at {FP32_LOGITS}"

In [ ]:
# peft is needed for LoRA adapters; viz for plots; mem pulls bitsandbytes for
# 8-bit AdamW. LoRA-QAT trains far fewer params than standard/scheduled QAT,
# but the frozen quantized base + activations still benefit from FP16 + grad
# checkpointing on a 14-16 GB T4.
!pip install -q -e ".[viz,mem]" peft
import torch
print(f"GPU: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'CPU'}")

## 2. Kaggle config overrides

LoRA-QAT uses `epochs=2` and a higher LR (2e-4) in the published config because only the adapters train. We keep those choices and just shrink seq_length / batch / accum to fit Kaggle.

In [ ]:
import yaml

DRIVE_MODEL_CACHE = "/content/drive/MyDrive/sqat-baseline/models/base"
LOCAL_MODEL_CACHE = f"{REPO_DIR}/models/base"
MODEL_CACHE       = DRIVE_MODEL_CACHE if Path(DRIVE_MODEL_CACHE).exists() else LOCAL_MODEL_CACHE

COLAB_CFG = Path(REPO_DIR) / "configs_colab" / "lora_qat"
COLAB_CFG.mkdir(parents=True, exist_ok=True)

# Same memory profile as notebook 03 — see that notebook for full rationale.
# 8-bit AdamW only matters for the LoRA adapter optimizer state (~16 MB), so it's
# a smaller absolute saving here, but BF16 base weights and gradient checkpointing
# still cut the activation/weight footprint roughly in half.
MEM_EFFICIENT = True

def patch_lora_config(src: str, dst: Path) -> Path:
    with open(src) as f:
        cfg = yaml.safe_load(f)
    cfg["model"]["cache_dir"] = MODEL_CACHE
    cfg["data"]["seq_length"] = 512
    cfg["training"]["epochs"] = 1
    cfg["training"]["batch_size"] = 4
    cfg["training"]["gradient_accumulation_steps"] = 8
    cfg["training"]["warmup_steps"] = 50
    if MEM_EFFICIENT:
        cfg["training"]["compute_dtype"] = "bf16"
        cfg["training"]["use_amp"] = False  # not needed when weights are already BF16
        cfg["training"]["use_8bit_optimizer"] = True
        cfg["training"]["gradient_checkpointing"] = True
    cfg["logging"]["save_every_steps"] = 999999
    cfg["logging"]["eval_every_steps"] = 500
    cfg["logging"]["log_dir"] = f"{REPO_DIR}/results/logs/{dst.stem}/"
    cfg["export"]["output_dir"] = f"{REPO_DIR}/models/gguf/"
    with dst.open("w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return dst

lora8_cfg = patch_lora_config("configs/lora_qat/lora_qat_int8.yaml", COLAB_CFG / "lora_qat_int8.yaml")
lora4_cfg = patch_lora_config("configs/lora_qat/lora_qat_int4.yaml", COLAB_CFG / "lora_qat_int4.yaml")
print("INT8 config:", lora8_cfg)
print("INT4 config:", lora4_cfg)
print(f"Model cache: {MODEL_CACHE}")
print(f"Memory-efficient training: {MEM_EFFICIENT}")

## 3. LoRA-QAT — INT8

Frozen quantized base + trainable rank-16 LoRA adapters on q/k/v/o projections.

In [ ]:
import gc
from src.training.trainer import run_experiment

results_int8 = run_experiment(
    config_path=str(lora8_cfg),
    device_str="cuda",
    baseline_logits=str(FP32_LOGITS),
    run_benchmarks=False,
)

print("\nLoRA-QAT INT8:")
for k, v in results_int8.items():
    print(f"  {k:25s}  {v}")

gc.collect(); torch.cuda.empty_cache()

## 4. LoRA-QAT — INT4

Same setup, INT4 base. The adapters compensate for the larger quantization error.

In [ ]:
results_int4 = run_experiment(
    config_path=str(lora4_cfg),
    device_str="cuda",
    baseline_logits=str(FP32_LOGITS),
    run_benchmarks=False,
)

print("\nLoRA-QAT INT4:")
for k, v in results_int4.items():
    print(f"  {k:25s}  {v}")

gc.collect(); torch.cuda.empty_cache()

## 5. Adapter parameter count

Quick sanity check on how many parameters were actually trained. For SmolLM2-1.7B with rank 16 on q/k/v/o, expect on the order of 4M trainable params (~0.25% of total). The exact number depends on hidden_size and num_attention_heads.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

adapter_dir = Path(REPO_DIR) / "models" / "checkpoints" / "lora_qat_int4" / "final_adapter"
if adapter_dir.exists():
    base = AutoModelForCausalLM.from_pretrained(
        "HuggingFaceTB/SmolLM2-1.7B", cache_dir=MODEL_CACHE, torch_dtype=torch.float16,
    )
    peft_model = PeftModel.from_pretrained(base, str(adapter_dir))
    total = sum(p.numel() for p in peft_model.parameters())
    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    print(f"Total params      : {total:,}")
    print(f"Trainable (LoRA)  : {trainable:,}")
    print(f"Trainable fraction: {trainable / total * 100:.3f}%")
    del peft_model, base
    torch.cuda.empty_cache()
else:
    print(f"Adapter dir not found: {adapter_dir}")

## 6. Comparison table

In [ ]:
import json
import pandas as pd

with open(Path(BASELINE_DIR) / "baseline_results.json") as f:
    fp32 = json.load(f)

rows = [
    {"method": "FP32",     "bits": 32, "perplexity": fp32["perplexity"], "kl_divergence": 0.0},
    {"method": "LoRA-QAT", "bits": 8,  "perplexity": results_int8.get("perplexity"), "kl_divergence": results_int8.get("kl_divergence")},
    {"method": "LoRA-QAT", "bits": 4,  "perplexity": results_int4.get("perplexity"), "kl_divergence": results_int4.get("kl_divergence")},
]
df = pd.DataFrame(rows)
df["ppl_delta_pct"] = ((df["perplexity"] / df["perplexity"].iloc[0]) - 1.0) * 100
df.round(4)

In [ ]:
summary_path = Path(REPO_DIR) / "results" / "lora_qat_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
with summary_path.open("w") as f:
    json.dump({"int8": results_int8, "int4": results_int4, "fp32": fp32},
              f, indent=2, default=str)
print(f"Wrote {summary_path}")

## 7. Training loss curves

In [ ]:
import json
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def read_jsonl(path):
    if not Path(path).exists():
        return []
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

mlog8 = read_jsonl(f"{REPO_DIR}/results/logs/lora_qat_int8/per_step_loss.jsonl")
mlog4 = read_jsonl(f"{REPO_DIR}/results/logs/lora_qat_int4/per_step_loss.jsonl")
elog8 = read_jsonl(f"{REPO_DIR}/results/logs/lora_qat_int8/training_steps.jsonl")
elog4 = read_jsonl(f"{REPO_DIR}/results/logs/lora_qat_int4/training_steps.jsonl")

fig = make_subplots(rows=1, cols=2, subplot_titles=("INT8", "INT4"),
                    specs=[[{"secondary_y": True}, {"secondary_y": True}]])
for col, (mlog, elog) in enumerate([(mlog8, elog8), (mlog4, elog4)], 1):
    if mlog:
        fig.add_trace(go.Scatter(
            x=[e["step"] for e in mlog], y=[e["loss"] for e in mlog],
            name="train loss (per step)",
            line=dict(color="#4C72B0", width=1), opacity=0.6,
            legendgroup=str(col), showlegend=(col == 1),
        ), row=1, col=col, secondary_y=False)
    if elog:
        fig.add_trace(go.Scatter(
            x=[e["step"] for e in elog], y=[e.get("val_ppl") for e in elog],
            name="val ppl",
            line=dict(color="#DD8452", dash="dash", width=2),
            mode="lines+markers", legendgroup=str(col), showlegend=(col == 1),
        ), row=1, col=col, secondary_y=True)

fig.update_xaxes(title_text="step")
fig.update_yaxes(title_text="loss", secondary_y=False, row=1, col=1)
fig.update_yaxes(title_text="val ppl", secondary_y=True, row=1, col=2)
fig.update_layout(title="LoRA-QAT — training curves (micro + macro)",
                  height=420, hovermode="x unified",
                  margin=dict(t=80, b=40, l=60, r=60))
fig.show()

## 8. Plotly comparison — quality vs FP32

In [ ]:
def plot_method_comparison(rows, title, fp32_ppl=None):
    labels = [r["label"] for r in rows]
    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=("Perplexity (lower=better)",
                                        "KL Divergence vs FP32 (lower=better)",
                                        "Model size (GB)"))
    fig.add_trace(go.Bar(x=labels, y=[r["perplexity"] for r in rows],
                         marker_color="#4C72B0",
                         text=[f"{r['perplexity']:.3f}" if r["perplexity"] is not None else "—" for r in rows],
                         textposition="outside"), 1, 1)
    if fp32_ppl is not None:
        fig.add_hline(y=fp32_ppl, line_dash="dash", line_color="black",
                      annotation_text=f"FP32 = {fp32_ppl:.3f}", row=1, col=1)
    fig.add_trace(go.Bar(x=labels, y=[r["kl_divergence"] for r in rows],
                         marker_color="#DD8452",
                         text=[f"{r['kl_divergence']:.4f}" if r["kl_divergence"] is not None else "—" for r in rows],
                         textposition="outside"), 1, 2)
    fig.add_trace(go.Bar(x=labels, y=[r["size_gb"] for r in rows],
                         marker_color="#55A868",
                         text=[f"{r['size_gb']:.2f}" for r in rows],
                         textposition="outside"), 1, 3)
    fig.update_layout(title=title, showlegend=False, height=420,
                      margin=dict(t=80, b=40, l=40, r=20))
    fig.show()

rows = [
    {"label": "FP32 baseline", "perplexity": fp32.get("perplexity"),         "kl_divergence": 0.0,                                 "size_gb": 6.5},
    {"label": "LoRA-QAT INT8", "perplexity": results_int8.get("perplexity"), "kl_divergence": results_int8.get("kl_divergence"),   "size_gb": 1.75},
    {"label": "LoRA-QAT INT4", "perplexity": results_int4.get("perplexity"), "kl_divergence": results_int4.get("kl_divergence"),   "size_gb": 0.87},
]
plot_method_comparison(rows, "LoRA-QAT — quantization quality vs FP32",
                       fp32_ppl=fp32.get("perplexity"))

## 9. Sample inference

Same prompts and helper as notebooks 02–04 — direct apples-to-apples comparison across notebooks.

LoRA-QAT inference reload uses `load_lora_checkpoint(base_model_name, adapter_dir, device, cache_dir)` which loads the FP32 base from cache and then applies the trained adapter on top.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.quantization.lora_qat import load_lora_checkpoint

SAMPLE_PROMPTS = [
    "The capital of France is",
    "Python is a programming language used for",
    "Once upon a time, in a small village,",
    "The chemical symbol for gold is",
    "To improve your sleep quality, you should",
]
MAX_NEW_TOKENS = 30
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-1.7B", cache_dir=MODEL_CACHE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def generate_with_model(model, prompts=SAMPLE_PROMPTS, max_new_tokens=MAX_NEW_TOKENS):
    model.eval()
    out = []
    for p in prompts:
        ids = tokenizer(p, return_tensors="pt").input_ids.to(device)
        with torch.no_grad():
            gen = model.generate(
                ids, max_new_tokens=max_new_tokens, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        out.append(tokenizer.decode(gen[0][ids.shape[1]:], skip_special_tokens=True).strip())
    return out

def free():
    gc.collect(); torch.cuda.empty_cache()

# 1. FP32 baseline
print("Generating FP32 ...")
fp32_model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-1.7B", cache_dir=MODEL_CACHE, torch_dtype=torch.float16,
).to(device)
fp32_outs = generate_with_model(fp32_model)
del fp32_model; free()

def load_lora_for_inference(adapter_dir):
    if not Path(adapter_dir).exists():
        print(f"  WARNING — adapter dir missing: {adapter_dir}")
        return None
    return load_lora_checkpoint(
        base_model_name="HuggingFaceTB/SmolLM2-1.7B",
        adapter_dir=adapter_dir,
        device=device,
        cache_dir=MODEL_CACHE,
    )

# 2. LoRA-QAT INT8
print("Generating LoRA-QAT INT8 ...")
m8 = load_lora_for_inference(f"{WORKING_DIR}/models/checkpoints/lora_qat_int8/final_adapter")
int8_outs = generate_with_model(m8) if m8 is not None else ["[adapter missing]"] * len(SAMPLE_PROMPTS)
del m8; free()

# 3. LoRA-QAT INT4
print("Generating LoRA-QAT INT4 ...")
m4 = load_lora_for_inference(f"{WORKING_DIR}/models/checkpoints/lora_qat_int4/final_adapter")
int4_outs = generate_with_model(m4) if m4 is not None else ["[adapter missing]"] * len(SAMPLE_PROMPTS)
del m4; free()

print("Done.")

In [ ]:
import pandas as pd
from IPython.display import display, HTML

inference_df = pd.DataFrame({
    "Prompt":          SAMPLE_PROMPTS,
    "FP32":            fp32_outs,
    "LoRA-QAT INT8":   int8_outs,
    "LoRA-QAT INT4":   int4_outs,
})

inference_df.to_json(Path(REPO_DIR) / "results" / "lora_qat_inference.json",
                     orient="records", indent=2)

display(HTML(inference_df.to_html(index=False, escape=False).replace(
    "<table", '<table style="font-family:monospace; font-size:12px"')))

## Export everything to Drive

LoRA adapters live in **directories** (PEFT format), not single `.pt` files. The export below copies the adapter dirs as `lora_qat_int{4,8}_adapter/` into Drive, plus the same data samples / metric summary / per-step + macro logs as notebooks 03 and 04.

In [ ]:
import shutil

dst = Path(DRIVE_ROOT)
(dst / "models" / "checkpoints").mkdir(parents=True, exist_ok=True)
(dst / "data_samples").mkdir(parents=True, exist_ok=True)

# 1. Copy adapter directories (PEFT format)
for label in ("int4", "int8"):
    src = Path(REPO_DIR) / "models" / "checkpoints" / f"lora_qat_{label}" / "final_adapter"
    out = dst / "models" / "checkpoints" / f"lora_qat_{label}_adapter"
    if src.exists():
        if out.exists():
            shutil.rmtree(out)
        shutil.copytree(src, out)
        size_mb = sum(f.stat().st_size for f in out.rglob("*") if f.is_file()) / 1e6
        print(f"  {out}/  ({size_mb:.0f} MB)")
    else:
        print(f"  SKIP — adapter not found: {src}")

# 2. Dataset samples
from src.utils.config_loader import load_config
from src.utils.data_loader import build_dataloaders, build_validation_loader

cfg = load_config(str(lora4_cfg))
train_loader, eval_loader = build_dataloaders(cfg, num_workers=0)
val_loader = build_validation_loader(cfg, num_workers=0)

def sample_loader(loader, n=8):
    chunks = []
    for batch in loader:
        chunks.append(batch["input_ids"][:n].cpu().clone())
        if sum(c.size(0) for c in chunks) >= n:
            break
    return torch.cat(chunks, dim=0)[:n]

torch.save({"input_ids": sample_loader(train_loader), "split": "train", "seq_length": cfg.data.seq_length},
           dst / "data_samples" / "train_sample.pt")
torch.save({"input_ids": sample_loader(val_loader),   "split": "validation", "seq_length": cfg.data.seq_length},
           dst / "data_samples" / "val_sample.pt")
torch.save({"input_ids": sample_loader(eval_loader),  "split": "test", "seq_length": cfg.data.seq_length},
           dst / "data_samples" / "test_sample.pt")
print(f"  Dataset samples written")

# 3. Metric summary CSV
metric_rows = []
for label, r in [("lora_qat_int8", results_int8), ("lora_qat_int4", results_int4)]:
    metric_rows.append({
        "experiment":      label,
        "perplexity":      r.get("perplexity"),
        "kl_divergence":   r.get("kl_divergence"),
        "final_loss":      r.get("final_loss"),
        "training_time_s": r.get("training_time_seconds"),
        "total_steps":     r.get("total_steps"),
    })
metric_df = pd.DataFrame(metric_rows).round(6)
metric_csv = dst / "results" / "metric_summary.csv"
metric_csv.parent.mkdir(parents=True, exist_ok=True)
metric_df.to_csv(metric_csv, index=False)
print(f"  Metric summary -> {metric_csv}")
display(metric_df)

# 4-5. Copy results dir + configs
shutil.copytree(f"{REPO_DIR}/results", dst / "results", dirs_exist_ok=True)
shutil.copytree(COLAB_CFG, dst / "configs_colab" / "lora_qat", dirs_exist_ok=True)

print(f"\nAll outputs saved to: {dst}")
!du -sh {dst}